# Adding detector-variation systematics (DetVar)

Covers adding detector variations to the systematics covariance, plus
a final conformance check before shipping a new analysis.

In [ ]:
%load_ext autoreload
%autoreload 2

import os, sys
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "../../../..")))

# TODO: set to your cafpyana clone (skipped if already exported).
os.environ.setdefault("CAFPYANA_PATH", "TODO_SET_ME/path/to/cafpyana")

import pandas as pd
import matplotlib.pyplot as plt
from cafpybara import core

# TODO: import your own analysis's name instead of _template.
import cafpybara.analyses._template as ana

STILL_ON_TEMPLATE = ana.__name__.endswith('_template')
if STILL_ON_TEMPLATE:
    print("Still importing _template directly -- see the TODO above.")
else:
    print("Imported cleanly -- a real, importable analysis.")

In [ ]:
# Define once, used throughout this notebook.
#
# MC_FILE needs real weights, e.g.
# `run_df_maker.py -c configs/hnl_mcnu_wgt.py -l <input.list> -o <output_name>`.
# GENIE/Flux/Geant4 weights included; cafpyana's own preselection runs too.
# DEFAULT_CUTS below is a separate, topology-specific cut on top.
#
# DATA_FILE/OFFBEAM_FILE use the same preselection-on config, e.g.
# `run_df_maker.py -c configs/hnl_data.py -l <input.list> -o <output_name>`.
#
# NOTE: run `concat_hdf.py` after producing per-shard dataframes.
# The resulting `file_idx` column is required to prevent duplication when computing `mcstat`.
MC_FILE = "your_mc_file.h5"
DATA_FILE = "your_data_file.h5"
OFFBEAM_FILE = "your_offbeam_file.h5"
VAR = "your_variable"
BINS = [0, 1, 2, 3]

## 1. Load MC + data

See below for the actual call:

In [ ]:
# What you'd actually write in your own analysis notebook:
try:
    offbeam_df, offbeam_pot, offbeam_ngates = ana.load_data(
        OFFBEAM_FILE, onbeam=False, offbeam_signal_value=ana.signal_dict["offbeam"],
        cuts=ana.DEFAULT_CUTS, preprocess_fn=ana.preprocess.preprocess_data,
    )
    data_df, data_pot, data_ngates = ana.load_data(
        DATA_FILE, cuts=ana.DEFAULT_CUTS, preprocess_fn=ana.preprocess.preprocess_data,
    )
    mc_df, mc_pot, mc_ngen = ana.load_mc(
        MC_FILE, cuts=ana.DEFAULT_CUTS, preprocess_fn=ana.preprocess.preprocess_mc,
    )

    f = 0.0725  # TODO: double check for the most-up-to-date fudge factor
    ongates_per_pot = data_ngates / data_pot
    scale_offbeam = ((1 - f) * (ongates_per_pot * mc_pot)) / offbeam_ngates

    weight_col = ('weights_mc', '', '', '', '', '')
    mc_df[weight_col] = 1.0
    offbeam_df[weight_col] = scale_offbeam

    mcstat_cols = ['__ntuple', 'entry', 'rec.slc..index', 'run', 'subrun', 'evt', 'sample', 'file_idx']
    mc_df = pd.concat([
        ana.mcstat(mc_df.assign(sample=0).set_index("sample", append=True).reset_index(), cols=mcstat_cols),
        offbeam_df.assign(sample=1).set_index("sample", append=True).reset_index(),
    ])
    print("Merged MC + off-beam rows:", len(mc_df))

    ana.plot_mc_data(mc_df, data_df, VAR, bins=BINS, scale=data_pot / mc_pot)  # no systs= yet
except Exception as e:
    print("Expected -- neither a real data file nor an \"offbeam\" category exist yet:")
    print(f"  {type(e).__name__}: {e}")

## 2. `detvar.py` -- register your analysis

* TODO: Set variables in `detvar.py`
  * `SLC_KEY`: table key for the slice-level df in raw `detvar_*.df` files.
  * `DEFAULT_PREPROCESS_FN`: same preprocess function you already use for real MC.
  * `SELECTION_FN_MAP`: selection-type name -> (cuts, stage); `"preprocess"` is injected automatically.

See hnlpi0 and nuecc examples.

See below for the sanity check:

In [ ]:
print("SLC_KEY:", ana.detvar.SLC_KEY)
print("SELECTION_FN_MAP keys:", list(ana.detvar.SELECTION_FN_MAP.keys()))
try:
    ana.detvar.DEFAULT_PREPROCESS_FN(None)
except Exception as e:
    print("Expected -- DEFAULT_PREPROCESS_FN not set yet:")
    print(f"  {type(e).__name__}: {e}")

## 3. Make detvar dataframe + set `config.py`

TODO:
* Produce raw `detvar_*.df` files: cafpyana's `run_df_maker.py -c
  configs/hnl_mcnu_detvar.py`.
* Turn those into `detvars.h5`: `core/detvar/process_detvars.py
  --analysis hnlpi0 -s preprocess`.
* Set `DETVAR_DIR`/`DETVAR_DICT_FILES` in `config.py` to point at that store.

See hnlpi0/nuecc `config.py`.

See below for the sanity check:

In [ ]:
print("DETVAR_DICT_FILES:", ana.config.DETVAR_DICT_FILES)

## 4. `funcs.py` -- adding `'detv'` to `systs_input()`

* Arguments:
  * add `'detv'` to `uncertainty_keys` to fold detector-systematic
    uncertainty into the covariance matrix.
  * needs `detvar_dict=`/`detvar_files=` -- `systs_input()` defaults
    `detvar_files` to `config.py`'s `DETVAR_DICT_FILES` from section 3 if
    you don't pass one.
* Under the hood: `get_total_cov` re-applies `cuts` to the CV/DV samples
  via `core.syst.get_detvar_systs`/`core.selection.select`. Pass the
  same `cuts` used to build `mc_df`, or the comparison mismatches
  selected vs. unselected populations.

See below for the actual call:

In [ ]:
# What you'd actually write in your own analysis notebook:
try:
    si = ana.systs_input(mc_pot, uncertainty_keys={'rate', 'norm', 'detv'})
    print("leaving detvar_dict/detvar_files unset falls back to config.py:")
    print("si.detvar_files is ana.config.DETVAR_DICT_FILES:", si.detvar_files is ana.config.DETVAR_DICT_FILES)

    detvar_syst_output = ana.get_total_cov(mc_df, reco_var=VAR, bins=BINS, **si.to_kwargs())
    print("detvar_syst_output.rate_cov shape:", detvar_syst_output.rate_cov.shape)
except Exception as e:
    print("Expected -- either no real MC file here yet, or DETVAR_DICT_FILES is still config.py's placeholder path:")
    print(f"  {type(e).__name__}: {e}")

In [ ]:
# plot_mc_data with systematics on (reuses detvar_syst_output from above):
try:
    ana.plot_mc_data(mc_df, data_df, VAR, bins=BINS, scale=data_pot / mc_pot, systs=detvar_syst_output)
except Exception as e:
    print("Expected -- either no real MC file here yet, or DETVAR_DICT_FILES is still config.py's placeholder path:")
    print(f"  {type(e).__name__}: {e}")

In [ ]:
# Systematics category breakdown
try:
    fig, axes, cats, cat_sums = ana.plot_syst_category_breakdown(
        syst_vars=[(detvar_syst_output, BINS, VAR)],
        category_dict=ana.category_dict_signal,
    )
    plt.show()
except Exception as e:
    print("Expected -- either no real MC file here yet, or DETVAR_DICT_FILES is still config.py's placeholder path:")
    print(f"  {type(e).__name__}: {e}")

## 5. Before you ship it: a manual conformance pass

* Run `scripts/verify_new_analysis.py <name>` -- automates this check,
  plus two this notebook can't: a no-op-preprocessing-default audit, and
  an `ast` sweep of example notebooks' `systs_input`/`SystematicsInput`/
  `PlottingConfig` calls for dead kwargs.
* If your loader needs real preprocessing, add it to that script's
  `NOOP_DEFAULT_CHECKS` table (curated per-analysis).
* Fuller checklist: README.md's "Reference: the override contract" and
  "Common mistakes" sections.

See below for the same idea, inline:

In [ ]:
def conformance_check(analysis_module):
    """Run this against your real analysis module, not the template."""
    problems = []

    for mod_name in ['io', 'funcs', 'plotting', 'selection', 'physics', 'syst', 'classes', 'preprocess', 'detvar']:
        core_mod = getattr(core, mod_name)
        missing = [n for n in getattr(core_mod, '__all__', []) if not hasattr(analysis_module, n)]
        if missing:
            problems.append(f"core.{mod_name} not fully re-exported: missing {missing}")

    if analysis_module.get_total_cov is not core.funcs.get_total_cov:
        problems.append("get_total_cov has been overridden -- delete it, keep only systs_input()")

    if not hasattr(analysis_module, 'systs_input'):
        problems.append("no systs_input() factory found")

    if problems:
        print(f"{len(problems)} problem(s) found:")
        for p in problems:
            print(f"  - {p}")
    else:
        print("No problems found by this (non-exhaustive) check.")
    return problems


conformance_check(ana)